# Phase 4 — What the co-mutation patterns tell us (Aim 2)

Phase 2 says *which* gene pairs co-occur or avoid each other; this notebook asks *why*, and
*where*, with one question per section -- each taken from the literature
(`docs/literature_review.md`) and tested on GENIE:

| # | question | literature behind it |
|---|---|---|
| 1 | Are exclusive pairs redundant hits in one pathway, and co-occurring pairs cooperating pathways? | Sanchez-Vega 2018; Mina 2017/2020; El Tekle 2021 |
| 2 | Which interactions are universal, and which change between cancer types? | Park & Lehner 2015 |
| 3 | Do different variants of the same gene have different partners? Is that smoking? | Scharpf 2022; Cook 2021; Vaeyens 2023 |
| 4 | Which combinations are rare -- and how rare? (the cases clinicians ask about) | -- |
| 5 | What is different in hypermutated tumours? | van de Haar 2019; Campbell 2017 |
| 6 | Do mutation-only and mutation-or-copy-number analyses agree? | -- |

**Inputs** (`data/processed/`): `comutation_pairs.parquet` (Phase 2 main analysis),
`comutation_pairs_hypermutated.parquet`, `comutation_pairs_mutation_only.parquet`,
`allele_pairs.parquet` / `allele_heterogeneity.parquet` / `allele_smoking_adjustment.parquet`
(`scripts/analyses/08_allele_screen.py`), plus the 10 TCGA oncogenic pathways
(`data/external/pathways/`, Sanchez-Vega et al. 2018, as used by cBioPortal) and CIViC.

**Outputs:** `cancer_type_heterogeneity.parquet`, `rare_combinations.parquet`.

## 0. Load

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2, fisher_exact, false_discovery_control
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"

pairs = pd.read_parquet(PROCESSED_DIR / "comutation_pairs.parquet")
hyper = pd.read_parquet(PROCESSED_DIR / "comutation_pairs_hypermutated.parquet")
mut_only = pd.read_parquet(PROCESSED_DIR / "comutation_pairs_mutation_only.parquet")
pathways = pd.read_csv(ROOT / "data" / "external" / "pathways" / "sanchez_vega_2018_pathways.tsv", sep="\t")
gene_pathways = pathways.groupby("gene")["pathway"].apply(set)
print(f"Phase 2: {len(pairs):,} pairs, {pairs['interaction_candidate'].sum():,} candidates, "
      f"{(pairs['interaction_candidate'] & pairs['replicated']).sum():,} replicated")
print(f"pathways: {pathways['pathway'].nunique()} pathways, {pathways['gene'].nunique()} genes")

Phase 2: 6,636 pairs, 1,543 candidates, 464 replicated
pathways: 10 pathways, 162 genes


## 1. Pathway structure of exclusivity and co-occurrence

**Claim:** mutual exclusivity reflects *redundancy* -- a second hit in an already-activated
pathway adds nothing (or harms the cell) -- so exclusive pairs should sit **within** a pathway;
co-occurrence reflects *cooperation* between pathways, so co-occurring pairs should sit
**across** pathways (Sanchez-Vega 2018; Mina 2020).

**Test:** among Phase 2 pairs whose two genes are both in the 10 TCGA pathways, compare how
often same-pathway and different-pathway pairs are candidates of each direction.

In [2]:
pw = pairs[pairs["Gene_A"].isin(gene_pathways.index) & pairs["Gene_B"].isin(gene_pathways.index)].copy()
pw["same_pathway"] = [bool(gene_pathways[a] & gene_pathways[b]) for a, b in zip(pw["Gene_A"], pw["Gene_B"])]
rows = []
for name, hit in [("exclusive candidate", pw["interaction_candidate"] & (pw["direction"] == "exclusive")),
                  ("co-occurring candidate", pw["interaction_candidate"] & (pw["direction"] == "co-occurring")),
                  ("exclusive, replicated", pw["interaction_candidate"] & pw["replicated"] & (pw["direction"] == "exclusive")),
                  ("co-occurring, replicated", pw["interaction_candidate"] & pw["replicated"] & (pw["direction"] == "co-occurring"))]:
    a, b = (hit & pw["same_pathway"]).sum(), (~hit & pw["same_pathway"]).sum()
    c, d = (hit & ~pw["same_pathway"]).sum(), (~hit & ~pw["same_pathway"]).sum()
    odds, p = fisher_exact([[a, b], [c, d]])
    rows.append({"pairs that are": name, "same pathway": f"{a}/{a + b} ({a / (a + b):.1%})",
                 "different pathways": f"{c}/{c + d} ({c / (c + d):.1%})", "odds ratio": round(odds, 2), "p": f"{p:.1e}"})
print(f"{len(pw):,} tested pairs with both genes in the 10 pathways; {pw['same_pathway'].sum():,} within one pathway")
pd.DataFrame(rows)

1,898 tested pairs with both genes in the 10 pathways; 382 within one pathway


,pairs that are,same pathway,different pathways,odds ratio,p
0,exclusive candidate,134/382 (35.1%),212/1516 (14.0%),3.32,2.9e-19
1,co-occurring candidate,22/382 (5.8%),89/1516 (5.9%),0.98,1.0e+00
2,"exclusive, replicated",68/382 (17.8%),69/1516 (4.6%),4.54,7.0e-16
3,"co-occurring, replicated",8/382 (2.1%),38/1516 (2.5%),0.83,8.5e-01


**Result.** Exclusivity is strongly concentrated **within** pathways (odds ratio ~3.3; ~4.5 for
replicated pairs) -- the redundancy model holds, and the top examples are the textbook ones
below. Co-occurrence, however, is **not** enriched across pathways (odds ratio ~1): in GENIE,
co-occurring pairs are no more likely to link different pathways than random pairs are. This
agrees with DISCOVER's conclusion that co-occurrence is the weaker, less structured signal,
and refines the "co-occurrence = cross-pathway cooperation" picture.

**Exclusive pairs across different pathways** are the interesting exceptions: El Tekle et
al. 2021 call these *incompatibilities* -- the two alterations drive divergent programmes
that do not work together in one cell.

In [3]:
show_cols = ["Cancer_Type", "Gene_A", "Gene_B", "n_both_altered", "expected_both", "fold_enrichment", "q_value"]
ex = pw[pw["interaction_candidate"] & pw["replicated"] & (pw["direction"] == "exclusive")].copy()
ex["pathways"] = [f"{'/'.join(sorted(gene_pathways[a]))} | {'/'.join(sorted(gene_pathways[b]))}"
                  for a, b in zip(ex["Gene_A"], ex["Gene_B"])]
print("Within one pathway (redundancy):")
display(ex[ex["same_pathway"]].sort_values("q_value")[show_cols + ["pathways"]].head(10))
print("Across pathways (incompatibility):")
display(ex[~ex["same_pathway"]].sort_values("q_value")[show_cols + ["pathways"]].head(10))

Within one pathway (redundancy):


,Cancer_Type,Gene_A,Gene_B,n_both_altered,expected_both,fold_enrichment,q_value,pathways
735,Non-Small Cell Lung Cancer,EGFR,KRAS,81,1085.432374,0.075051,1.424194e-319,RTK-RAS | RTK-RAS
1116,Colorectal Cancer,BRAF,KRAS,59,374.378108,0.158718,2.298756e-97,RTK-RAS | RTK-RAS
4814,Melanoma,BRAF,NRAS,17,203.618336,0.085735,9.316754e-76,RTK-RAS | RTK-RAS
869,Non-Small Cell Lung Cancer,MDM2,TP53,104,375.899530,0.277631,7.614878e-64,Cell Cycle/TP53 | Cell Cycle/TP53
1262,Colorectal Cancer,KRAS,NRAS,29,225.306936,0.130643,1.506227e-62,RTK-RAS | RTK-RAS
1664,Glioma,EGFR,NF1,71,283.622946,0.251652,1.810808e-54,RTK-RAS | RTK-RAS
3144,Endometrial Cancer,PIK3CA,PIK3R1,127,359.868409,0.353805,5.798369e-51,PI3K | PI3K
1592,Glioma,CDKN2A,TP53,536,883.981974,0.606570,1.267556e-45,Cell Cycle/TP53 | Cell Cycle/TP53
1767,Glioma,MDM2,TP53,8,125.364304,0.067533,8.575462e-43,Cell Cycle/TP53 | Cell Cycle/TP53
842,Non-Small Cell Lung Cancer,KRAS,MET,60,224.229272,0.269213,7.574579e-39,RTK-RAS | RTK-RAS


Across pathways (incompatibility):


,Cancer_Type,Gene_A,Gene_B,n_both_altered,expected_both,fold_enrichment,q_value,pathways
748,Non-Small Cell Lung Cancer,EGFR,STK11,47,482.439146,0.098356,1.230157e-152,RTK-RAS | PI3K
733,Non-Small Cell Lung Cancer,EGFR,KEAP1,47,365.801319,0.129675,1.644463e-102,RTK-RAS | NRF2
539,Breast Cancer,PIK3CA,TP53,1447,1964.089163,0.736795,3.236982e-46,PI3K | Cell Cycle/TP53
855,Non-Small Cell Lung Cancer,KRAS,TP53,1041,1493.017411,0.697347,1.479929e-43,RTK-RAS | Cell Cycle/TP53
1674,Glioma,EGFR,TP53,349,641.589834,0.544316,7.438930e-43,RTK-RAS | Cell Cycle/TP53
4405,Bladder Cancer,FGFR3,TP53,89,265.334062,0.336676,4.413011e-40,RTK-RAS | Cell Cycle/TP53
992,Colorectal Cancer,APC,BRAF,314,589.119714,0.533395,4.648966e-38,WNT | RTK-RAS
303,Breast Cancer,ERBB2,PTEN,40,180.977481,0.223168,1.652071e-36,RTK-RAS | PI3K
3053,Endometrial Cancer,CTNNB1,TP53,21,132.975551,0.161078,1.282036e-33,WNT | Cell Cycle/TP53
959,Non-Small Cell Lung Cancer,STK11,TP53,409,672.343688,0.608611,4.745242e-30,PI3K | Cell Cycle/TP53


## 2. Universal versus cancer-type-specific interactions

**Claim:** genetic interactions are rewired between cell types (Park & Lehner 2015: 30 of 52
re-testable interactions differed between cancer types).

**Test:** for every pair tested in >= 3 cancer types, compare its effect -- the log of
observed / expected co-alteration -- across cancer types with Cochran's Q (inverse-variance
weights; variance of a log observed/expected ratio ~ 1 / observed). BH-FDR across pairs.
Each cancer type's effect is already adjusted for alteration load and subtype (Phase 2), so
differences between types are not driven by different mutation rates.

In [4]:
pairs["key"] = pairs["Gene_A"] + " / " + pairs["Gene_B"]
pairs["theta"] = np.log((pairs["n_both_altered"] + 0.5) / (pairs["expected_both"] + 0.5))
pairs["var"] = 1 / (pairs["n_both_altered"] + 0.5)
rows = []
for key, g in pairs.groupby("key"):
    if len(g) < 3:
        continue
    w = 1 / g["var"]; pooled = (w * g["theta"]).sum() / w.sum()
    Q = (w * (g["theta"] - pooled) ** 2).sum(); df = len(g) - 1
    sig = g[g["q_value"] < 0.05]
    rows.append({"pair": key, "n_cancer_types": len(g), "pooled_fold": np.exp(pooled), "Q": Q, "p": chi2.sf(Q, df),
                 "I2": max(0.0, (Q - df) / Q) if Q > 0 else 0.0,
                 "co_occurring_in": ", ".join(sig.loc[sig["direction"] == "co-occurring", "Cancer_Type"]),
                 "exclusive_in": ", ".join(sig.loc[sig["direction"] == "exclusive", "Cancer_Type"]),
                 "n_sig_co": (sig["direction"] == "co-occurring").sum(), "n_sig_ex": (sig["direction"] == "exclusive").sum()})
het = pd.DataFrame(rows)
het["q"] = false_discovery_control(het["p"])
het["pattern"] = np.select(
    [(het["n_sig_co"] > 0) & (het["n_sig_ex"] > 0), het["q"] < 0.05, (het["n_sig_co"] + het["n_sig_ex"]) >= 3],
    ["opposite directions", "cancer-type-specific", "universal"], "not significant / not enough")
het.to_parquet(PROCESSED_DIR / "cancer_type_heterogeneity.parquet", index=False)
print(f"{len(het):,} pairs tested in >= 3 cancer types; heterogeneous (q < 0.05): {(het['q'] < 0.05).sum():,} "
      f"({(het['q'] < 0.05).mean():.0%})")
het["pattern"].value_counts()

539 pairs tested in >= 3 cancer types; heterogeneous (q < 0.05): 165 (31%)


pattern
not significant / not enough    351
cancer-type-specific            121
opposite directions              45
universal                        22
Name: count, dtype: int64

In [5]:
print("Significant in OPPOSITE directions in different cancer types:")
display(het[het["pattern"] == "opposite directions"].sort_values("q")
        [["pair", "n_cancer_types", "co_occurring_in", "exclusive_in", "q"]].head(12))
print("Universal (same direction, significant in >= 3 cancer types, not heterogeneous):")
display(het[het["pattern"] == "universal"].sort_values("n_cancer_types", ascending=False)
        [["pair", "n_cancer_types", "pooled_fold", "n_sig_co", "n_sig_ex", "I2"]].round(2).head(12))

Significant in OPPOSITE directions in different cancer types:


,pair,n_cancer_types,co_occurring_in,exclusive_in,q
216,CDKN2A / TP53,31,"Non-Small Cell Lung Cancer, Pancreatic Cancer, Head and Neck Cance...","Mature B-Cell Neoplasms, Glioma, Soft Tissue Sarcoma, Bladder Canc...",6.418252e-40
292,EGFR / TP53,8,Non-Small Cell Lung Cancer,Glioma,8.375165e-39
445,KRAS / TP53,20,Pancreatic Cancer,"Non-Small Cell Lung Cancer, Colorectal Cancer, Ovarian Cancer, End...",2.111404e-26
157,CCND1 / TP53,9,Head and Neck Cancer,Breast Cancer,6.494555e-20
386,FOXA1 / TP53,4,Non-Small Cell Lung Cancer,"Breast Cancer, Prostate Cancer",3.511513e-18
509,PIK3R1 / PTEN,3,"Colorectal Cancer, Endometrial Cancer",Glioma,1.380055e-17
514,PTEN / TP53,19,"Prostate Cancer, Thyroid Cancer","Colorectal Cancer, Glioma, Endometrial Cancer, Head and Neck Cance...",6.816679e-15
262,CTNNB1 / KRAS,8,Colorectal Cancer,"Non-Small Cell Lung Cancer, Endometrial Cancer",3.162566e-14
358,FGF19 / TP53,7,Head and Neck Cancer,Breast Cancer,4.930127e-14
375,FGF4 / TP53,6,Head and Neck Cancer,Breast Cancer,2.446811e-13


Universal (same direction, significant in >= 3 cancer types, not heterogeneous):


,pair,n_cancer_types,pooled_fold,n_sig_co,n_sig_ex,I2
88,ATM / TP53,17,0.52,0,9,0.27
245,CDKN2B / RB1,12,0.25,0,10,0.51
210,CDKN2A / RB1,12,0.25,0,10,0.36
512,PTEN / RB1,9,1.38,3,0,0.41
152,CCND1 / PIK3CA,9,0.75,0,3,0.00
453,MDM2 / RB1,6,0.50,0,5,0.44
498,PIK3CA / PIK3R1,5,0.41,0,3,0.51
467,MTAP / RB1,5,0.22,0,4,0.23
450,MDM2 / NF1,5,0.53,0,3,0.62
306,ERBB2 / MDM2,5,0.65,0,3,0.65


**Result.** About a third of the interactions that can be compared across cancer types
change between them -- in line with Park & Lehner. The opposite-direction pairs read as
biology, not noise:
- **EGFR / TP53** -- co-occurring in lung cancer (EGFR-mutant adenocarcinomas usually carry
  TP53 mutations), exclusive in glioma (EGFR-amplified glioblastoma vs TP53-mutant astrocytoma).
- **KRAS / TP53** -- co-occurring in pancreatic cancer (the canonical progression KRAS -> TP53),
  exclusive in lung and colorectal cancer.
- **CCND1 / TP53** -- co-occurring in head and neck cancer (11q13 amplification in TP53-mutant,
  HPV-negative tumours), exclusive in breast cancer (luminal CCND1 vs basal TP53).

The universal pairs are the cell-cycle and DNA-damage redundancies (*ATM*/*TP53*,
*CDKN2A*/*RB1*, *MDM2*/*TP53*) that hold in almost every tissue -- the ones a clinician can
rely on regardless of cancer type.

## 3. Variant-level partners, and the smoking confounder

**Claim:** co-mutation is allele-specific (Scharpf 2022 in GENIE; Cook 2021).
**Test** (`scripts/analyses/08_allele_screen.py`): hotspot genes are split into allele groups
(KRAS G12C/G12D/G12V/..., BRAF V600E vs non-V600, EGFR L858R vs exon 19 deletion, PIK3CA
helical vs kinase domain, IDH1 R132H), each allele is tested against every partner with the
Phase 2 conditional test, and Cochran's Q asks whether alleles of the same gene differ.

In [6]:
allele_het = pd.read_parquet(PROCESSED_DIR / "allele_heterogeneity.parquet")
print(f"(gene, partner) combinations with >= 2 alleles tested: {len(allele_het)}; "
      f"alleles differ (q < 0.05): {(allele_het['q'] < 0.05).sum()}")
allele_het.sort_values("q").head(12)[["Cancer_Type", "gene", "partner", "folds", "q"]]

(gene, partner) combinations with >= 2 alleles tested: 199; alleles differ (q < 0.05): 10


,Cancer_Type,gene,partner,folds,q
152,Non-Small Cell Lung Cancer,EGFR,RBM10,L858R 2.01; ex19del 0.55,3.712841e-22
17,Breast Cancer,PIK3CA,GATA3,helical 0.76; kinase 0.42,5.636852e-06
186,Pancreatic Cancer,KRAS,ARID1A,G12D 1.16; G12R 0.59; G12V 0.82; Q61H 0.92,3.729327e-03
160,Non-Small Cell Lung Cancer,KRAS,ATM,G12C 2.20; G12D 1.02; G12V 1.74,7.161163e-03
127,Melanoma,BRAF,PREX2,V600E 0.40; nonV600 1.67,8.237854e-03
125,Melanoma,BRAF,NRAS,V600E 0.04; nonV600 0.25,8.464316e-03
38,Colorectal Cancer,KRAS,BCL2L1,G12C 0.32; G12D 0.33; G12V 0.31; G13D 0.65,1.773026e-02
10,Breast Cancer,PIK3CA,ERBB2,helical 0.61; kinase 0.84,2.113835e-02
52,Colorectal Cancer,KRAS,SMAD4,G12C 1.13; G12D 1.20; G12V 1.25; G13D 0.82,2.113835e-02
28,Breast Cancer,PIK3CA,PTEN,helical 0.75; kinase 0.47,2.113835e-02


**The smoking confounder.** KRAS G12C is a C>A substitution, the signature of tobacco
smoke, and STK11 / KEAP1 mutations are also smoking-associated -- so the classic "G12C goes with
STK11 and KEAP1" could partly be *two consequences of smoking* rather than an interaction.
Each patient's smoking-signature level was estimated from their own mutation spectrum (share
of C>A substitutions, excluding the genes tested here, so a G12C cannot raise its own score),
and the test was repeated with gene rates fitted within smoking-signature tertiles as well as
subtypes.

In [7]:
pd.read_parquet(PROCESSED_DIR / "allele_smoking_adjustment.parquet").round(3)

,allele,partner,fold,q,fold_smoking_adjusted,q_smoking_adjusted
0,KRAS_G12C,STK11,1.973,0.000,1.726,0.000
1,KRAS_G12C,KEAP1,1.677,0.000,1.491,0.000
2,KRAS_G12C,SMARCA4,1.396,0.003,1.255,0.043
3,KRAS_G12C,TP53,0.671,0.000,0.668,0.000
4,KRAS_G12C,ATM,2.196,0.000,1.793,0.000
5,KRAS_G12C,RBM10,1.225,0.027,1.089,0.395
6,KRAS_G12D,STK11,1.402,0.003,1.401,0.003
7,KRAS_G12D,KEAP1,1.334,0.033,1.333,0.034
8,KRAS_G12D,SMARCA4,1.229,0.315,1.236,0.307
9,KRAS_G12D,TP53,0.612,0.000,0.614,0.000


**Result.** Adjusting for the smoking signature shrinks the G12C associations (STK11
~1.97 -> ~1.73, KEAP1 ~1.68 -> ~1.49) but leaves them strong and significant, while the
non-smoking allele G12D is unchanged -- the expected negative control. So about a fifth of the
G12C-STK11 association (on the log scale) is shared smoking exposure and the rest is specific
to G12C; the weaker G12C-RBM10 association disappears entirely (1.23 -> 1.09, not significant)
-- it is explained by smoking. Other allele-level differences: *EGFR* L858R vs exon 19 deletion differ
sharply in *RBM10* co-mutation; *PIK3CA* kinase-domain vs helical mutations differ in *GATA3*
and *PTEN* (breast).

## 4. Rare combinations: how rare, and do they exist at all?

Clinicians ask about combinations they rarely see. The strongly exclusive candidates are
exactly those: below are the replicated exclusive pairs with the fewest patients relative to
expectation, with how many such patients exist in GENIE (they are real patients the lookup
tool can show) and whether CIViC holds evidence for the combination.

In [8]:
civic = pd.read_csv(ROOT / "data" / "external" / "civic" / "nightly-ClinicalEvidenceSummaries.tsv", sep="\t")
civic_combos = civic.loc[civic["molecular_profile"].str.contains(" AND ", na=False), "molecular_profile"].unique()
combo_genes = [set(p.split(" ")[0] for p in [x.strip() for x in mp.split(" AND ")]) for mp in civic_combos]

rare = pairs[pairs["interaction_candidate"] & pairs["replicated"] & (pairs["direction"] == "exclusive")].copy()
rare["patients_with_both"] = rare["n_both_altered"]
rare["times_rarer_than_expected"] = (rare["expected_both"] / rare["n_both_altered"].clip(lower=0.5)).round(1)
rare["civic_combination_evidence"] = [any({a, b} <= g for g in combo_genes) for a, b in zip(rare["Gene_A"], rare["Gene_B"])]
rare = rare.sort_values("times_rarer_than_expected", ascending=False)
rare[["Cancer_Type", "Gene_A", "Gene_B", "patients_with_both", "expected_both", "times_rarer_than_expected",
      "n_tested_both", "q_value", "civic_combination_evidence"]].to_parquet(PROCESSED_DIR / "rare_combinations.parquet", index=False)
print(f"{len(rare)} replicated exclusive pairs; {(rare['patients_with_both'] == 0).sum()} never seen together; "
      f"{(rare['patients_with_both'].between(1, 20)).sum()} seen in 1-20 patients")
rare[["Cancer_Type", "Gene_A", "Gene_B", "patients_with_both", "expected_both", "times_rarer_than_expected",
      "civic_combination_evidence"]].round(1).head(20)

336 replicated exclusive pairs; 3 never seen together; 116 seen in 1-20 patients


,Cancer_Type,Gene_A,Gene_B,patients_with_both,expected_both,times_rarer_than_expected,civic_combination_evidence
5005,Melanoma,GNA11,GNAQ,0,44.1,88.1,False
1712,Glioma,IDH1,MDM2,0,34.0,68.1,False
1693,Glioma,H3F3A,IDH1,1,42.6,42.6,False
3305,Cancer of Unknown Primary,CCND1,RB1,0,13.9,27.8,False
5260,Thyroid Cancer,BRAF,HRAS,1,19.4,19.4,False
3728,Hepatobiliary Cancer,IDH1,SMAD4,1,18.2,18.2,False
1767,Glioma,MDM2,TP53,8,125.4,15.7,False
5261,Thyroid Cancer,BRAF,NRAS,4,58.6,14.7,True
3004,Endometrial Cancer,CCNE1,CTNNB1,1,14.3,14.3,False
4554,Bladder Cancer,MTAP,RB1,3,41.4,13.8,False


## 5. What is different in hypermutated tumours?

Hypermutated tumours (TMB 10-100) were analysed on their own in Phase 2. Which significant
pairs there have no counterpart in non-hypermutated tumours of the same cancer type?

In [9]:
h = hyper[hyper["q_value"] < 0.05].merge(
    pairs[["Cancer_Type", "Gene_A", "Gene_B", "q_value", "direction"]], on=["Cancer_Type", "Gene_A", "Gene_B"],
    how="left", suffixes=("", "_nonhyper"))
h["status_in_nonhyper"] = np.select([h["q_value_nonhyper"] < 0.05, h["q_value_nonhyper"].notna()],
                                    ["significant too", "tested, not significant"], "not tested (gene not frequent)")
print(f"{len(h):,} significant pairs in hypermutated tumours:")
display(pd.crosstab(h["Cancer_Type"], h["status_in_nonhyper"]).sort_values("not tested (gene not frequent)", ascending=False).head(8))
h[h["Cancer_Type"] == "Colorectal Cancer"].sort_values("q_value")[
    ["Gene_A", "Gene_B", "n_both_altered", "expected_both", "direction", "q_value", "status_in_nonhyper"]].head(10)

881 significant pairs in hypermutated tumours:


status_in_nonhyper,not tested (gene not frequent),significant too,"tested, not significant"
Cancer_Type,,,
Colorectal Cancer,583,35,20
Non-Small Cell Lung Cancer,47,25,0
Melanoma,29,16,6
Mature B-Cell Neoplasms,23,3,0
Bladder Cancer,22,39,2
Endometrial Cancer,8,4,1
Breast Cancer,1,10,0
Cancer of Unknown Primary,0,3,0


,Gene_A,Gene_B,n_both_altered,expected_both,direction,q_value,status_in_nonhyper
256,BRAF,KRAS,58,284.759586,exclusive,5.658536e-68,significant too
176,APC,RNF43,230,486.334304,exclusive,2.275517e-51,not tested (gene not frequent)
524,KRAS,RNF43,162,360.018294,exclusive,1.152376e-37,not tested (gene not frequent)
118,APC,BRAF,212,385.117608,exclusive,3.031972e-25,significant too
208,ARID1A,TP53,255,413.393601,exclusive,1.086000e-19,"tested, not significant"
668,RNF43,TP53,225,372.860388,exclusive,1.362784e-18,not tested (gene not frequent)
467,KMT2B,TP53,160,286.269318,exclusive,1.693187e-18,not tested (gene not frequent)
706,TCF7L2,TP53,185,313.515425,exclusive,5.477571e-16,significant too
481,KMT2D,TP53,261,403.684926,exclusive,8.665004e-16,not tested (gene not frequent)
689,SMAD4,TGFBR2,15,74.843617,exclusive,1.855439e-14,not tested (gene not frequent)


**Result.** Most hypermutated-only signal involves genes that are frequent *only* under
hypermutation, and in colorectal cancer it is dominated by mismatch-repair target genes
(*RNF43*, *KMT2D*, *ARID1A*, *MSH3*) avoiding the chromosomal-instability drivers (*TP53*,
*APC*, *KRAS*): two tumour subtypes that share a TMB range (MSI vs hypermutated MSS), not gene
interactions. This is why the hypermutated group is reported as a description and kept out of
the interaction candidates.

## 6. Do the mutation-only and mutation-or-copy-number analyses agree?

In [10]:
m = mut_only.merge(pairs[["Cancer_Type", "Gene_A", "Gene_B", "q_value", "direction"]],
                  on=["Cancer_Type", "Gene_A", "Gene_B"], suffixes=("_mut", "_any"))
both_sig = (m["q_value_mut"] < 0.05) & (m["q_value_any"] < 0.05)
print(f"pairs in both analyses: {len(m):,}; significant in both: {both_sig.sum():,}; "
      f"same direction: {(m.loc[both_sig, 'direction_mut'] == m.loc[both_sig, 'direction_any']).mean():.1%}")
print(f"significant in mutation-only but not in mutation-or-copy-number: "
      f"{((m['q_value_mut'] < 0.05) & (m['q_value_any'] >= 0.05)).sum():,}; the reverse: "
      f"{((m['q_value_mut'] >= 0.05) & (m['q_value_any'] < 0.05)).sum():,}")

pairs in both analyses: 2,193; significant in both: 376; same direction: 98.9%
significant in mutation-only but not in mutation-or-copy-number: 179; the reverse: 158


## 7. A blind spot we found while building this: the amplifier-burden confound

Phase 2's conditional test controls for each patient's *total* number of altered genes, so a
heavily altered tumour can't manufacture a fake pair just by having many hits (Section 4a of
Phase 2). But it does **not** control for *which* genes cluster together within that total --
and some genes are altered almost only as passengers on a genomic amplicon (a stretch of
chromosome gained all at once), not through any process of their own. A patient with one such
passenger gene amplified is usually a broadly "amplifier" tumour, likely to have *several*
other genes amplified too, anywhere in the genome. That can look exactly like two genes
"co-occurring" when really both are downstream of the same generic instability.

**Case study: CD79B in breast cancer.** *CD79B* is a B-cell receptor gene with no established
role in breast cancer -- it has no business appearing at all. Phase 2 flagged it as
co-occurring with *GATA3*, *MDM2*, *AURKA* and *GNAS* (q < 1e-5 for all four, all replicated
outside MSK). 99% of CD79B's alterations are amplifications, not mutations. It sits at 17q23,
next to *PPM1D* and *TBX2* -- a real, recurrent breast-cancer amplicon -- but GENIE panels
almost never measure PPM1D or TBX2 for copy number (0% of breast patients), so CD79B is the
only visible marker of that amplicon. Patients with CD79B amplified carry a **median of 7**
other amplified genes, against **1** for a typical breast-cancer patient -- they are simply
"amplifier" tumours.

In [11]:
import statsmodels.api as sm

bc = clinical_tidy = pd.read_parquet(PROCESSED_DIR / "clinical_tidy.parquet")
bc = bc[(bc.CANCER_TYPE == "Breast Cancer") & bc.IS_REPRESENTATIVE & bc.CNA_TESTED].set_index("SAMPLE_ID")
alterations_long = pd.read_parquet(PROCESSED_DIR / "alterations_long.parquet",
                                   columns=["Sample_ID", "Hugo_Symbol", "Alteration_Type", "CNA_Direction", "CANCER_TYPE"])
amp = alterations_long[(alterations_long.CANCER_TYPE == "Breast Cancer") & (alterations_long.Alteration_Type == "CNV")
                       & (alterations_long.CNA_Direction == "Amplification")]
burden = amp.groupby("Sample_ID").size().reindex(bc.index).fillna(0)


def flag(gene):
    return bc.index.isin(set(amp.loc[amp.Hugo_Symbol == gene, "Sample_ID"])).astype(int)


d = pd.DataFrame(index=bc.index)
d["cd79b"] = flag("CD79B")
d["log_burden_excl"] = np.log1p(burden.values - d["cd79b"])   # every OTHER gene's amplification burden

rows = []
for partner in ["GATA3", "MDM2", "AURKA", "GNAS"]:
    d["y"] = flag(partner)
    raw = pairs[(pairs.Cancer_Type == "Breast Cancer") & pairs.Gene_A.isin(["CD79B", partner])
               & pairs.Gene_B.isin(["CD79B", partner])].iloc[0]
    m = sm.Logit(d["y"], sm.add_constant(d[["cd79b", "log_burden_excl"]])).fit(disp=0)
    rows.append({"partner": partner, "Phase 2 fold (raw)": round(raw.fold_enrichment, 2),
                 "odds ratio after adjusting for amp burden": round(np.exp(m.params["cd79b"]), 2),
                 "p (adjusted)": round(m.pvalues["cd79b"], 4)})
pd.DataFrame(rows)

,partner,Phase 2 fold (raw),odds ratio after adjusting for amp burden,p (adjusted)
0,GATA3,2.63,0.41,0.0022
1,MDM2,2.44,2.69,0.0000
2,AURKA,2.14,2.19,0.0000
3,GNAS,1.96,2.04,0.0000


**Result:** after controlling for each patient's overall amplification burden --

| partner | raw co-occurrence (Phase 2) | after adjusting for amp. burden |
|---|---|---|
| GATA3 | 2.63-fold *together* | **reverses** to 0.41 (less likely together) |
| MDM2 | 2.44-fold together | 2.69 -- survives |
| AURKA | 2.14-fold together | 2.19 -- survives |
| GNAS | 1.96-fold together | 2.04 -- survives |

**GATA3's association was a pure confound** -- it flips sign once burden is accounted for.
MDM2, AURKA and GNAS survive, which is itself interesting: it suggests a *specific* pattern of
co-amplified regions (12q15/*MDM2*, 20q13/*AURKA*+*GNAS*) in a subset of highly rearranged
tumours, not just generic noise.

**How widespread is this risk?** Of the 128 replicated co-occurring candidates, **24 (19%)**
are driven almost entirely (>= 80%) by both genes being *amplified* rather than mutated in the
same patients -- the same shape of risk as CD79B, on genes scattered across different
chromosomes (e.g. *CDK4*/*KIT*/*KDR*/*PDGFRA* in glioma; *MYC*/*TERT* and *MDM2*/*TERT* in lung
cancer). **Mutation-driven co-occurring pairs are not at this risk**, because point mutations
don't cluster by chromosome position the way amplifications do.

**Practical takeaway:** when the lookup tool or this list reports a co-occurring pair, check
`frac_CNV_CNV` (in `mechanism_pairs.parquet`) -- a pair that is mostly copy-number should be
treated more cautiously than a mutation-driven one, unless (like MDM2/AURKA/GNAS above) it
survives a burden check. A systematic burden-adjusted re-test of all CNV-heavy candidates is a
natural next step, not done here for time.

## 8. An exception to the pathway rule: cooperating, not redundant

Section 1 found that exclusive pairs cluster strongly within one TCGA pathway (same-pathway
redundancy). *NF1* + *PTPN11* in glioma is a clean exception: both sit in the RTK-RAS pathway,
yet they **co-occur** (100 patients vs 33 expected, q = 4e-20, replicated), and 96 of those 100
co-alterations are two separate point mutations -- not an amplicon, so Section 7's confound
does not apply here.

Biologically this makes sense rather than contradicts the rule: *NF1* is a brake on RAS
signalling (losing it releases the brake), while *PTPN11* (SHP2) is an accelerator that works
with growth-factor receptors to switch RAS on. Losing the brake and pressing the accelerator
are not redundant -- they are two different ways of escalating the same pathway, which fits
a cooperating, not a redundant, pair.

## Discussion

1. **Exclusivity is pathway redundancy; co-occurrence is not pathway cooperation.** Exclusive
   pairs are 3-4x enriched within the same TCGA pathway, co-occurring pairs are not enriched
   across pathways -- consistent with "biology drives exclusivity, chance explains much
   co-occurrence" (Canisius 2016), and a reason to trust exclusive calls more in the clinic.
2. **A third of comparable interactions depend on the cancer type** (Park & Lehner 2015), some
   reversing direction (EGFR/TP53, KRAS/TP53, CCND1/TP53). The lookup tool therefore always
   answers within one cancer type.
3. **Partners depend on the variant** (EGFR L858R vs exon 19; PIK3CA helical vs kinase; KRAS
   alleles), and part of a famous allele association is shared exposure: smoking explains about
   a fifth of KRAS G12C-STK11 and all of G12C-RBM10 -- a confounder the literature mentions but
   rarely adjusts for.
4. **Rare combinations exist and can be counted**: the strongest exclusivities are 10-100x
   rarer than expected (GNAQ + GNA11 in melanoma: 0 vs 44 expected; H3F3A + IDH1 in glioma:
   1 vs 43, two distinct glioma classes), and a few have combination-level clinical evidence in
   CIViC (e.g. BRAF V600E + NRAS: resistance to BRAF inhibitors).
5. **Hypermutated tumours mainly add subtype structure**, not new interactions.

**Limits:** pathway membership uses 10 curated pathways (165 genes) -- pairs outside them are
not classified; heterogeneity variances are approximate (Poisson); the smoking proxy is
inferred from panel mutation spectra (no smoking history in GENIE); mutual exclusivity in a
cross-section cannot by itself separate redundancy from synthetic lethality.